In [2]:
"""
Figure S3 — GLORYS12 Validation: Fram Strait Moorings
=============================================
Supplementary Figure S3: GLORYS12 validation against Laura de Steur's
Fram Strait mooring timeseries (NPI eastern array, EGC side).

WORKFLOW
--------
1. Load / collocate: for each mooring extract GLORYS12 T and S at the
   nearest horizontal grid point, interpolated vertically to the mean
   instrument depth. Obs resampled to monthly means (≥90 % coverage
   required). Results cached to CSV.

2. Scatter figures (reference only — not used in paper):
   One 2-panel T/S scatter per mooring saved to outputs/figures/.

3. Statistics: bias (GLORYS − obs) and RMSE computed per mooring per
   variable, saved to mooring_validation_stats.csv.

4. Bar-chart figure (SUPPLEMENTARY FIG S3):
   Grouped bar chart of bias and RMSE for T and S at each mooring,
   ordered west to east (F17 → F14 → F13 → F12 → F11).

Moorings: F11, F12, F13, F14, F17
Data source: timeseries_F{XX}_{depth}_{start}_{end}.mat (Laura de Steur / NPI)

MATLAB datenum conversion
--------------------------
Anchor: MATLAB datenum 719529 == 1970-01-01 (Unix epoch). Vectorised
float arithmetic used to avoid integer overflow.

In-situ T → potential temperature via gsw.pt_from_t (ref pressure 0).

GLORYS collocation
------------------
- Nearest neighbour in lat/lon
- Vertical: np.interp to mean instrument pressure (dbar treated as m)
- Monthly: match by year + month

=============================================================================
Version:       1.0.0
Last Modified: 22-06-2026
Author:        Chris Barrell
=============================================================================
OUTPUT
=============================================================================
./outputs/figures/fig_supp_mooring_scatter_F{XX}.png    (reference, one per mooring)
./outputs/figures/figure_s03_FS_mooring_glorys_verification.png  (supplementary figure)
./outputs/processed_data/figure_s03/glorys_mooring_collocated.csv
./outputs/processed_data/figure_s03/mooring_validation_stats.csv

=============================================================================
"""

import sys
import os
import glob
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import scipy.io as sio
import xarray as xr
from pathlib import Path
from scipy import stats
import gsw

warnings.filterwarnings("ignore")

# ---------------------------------------------------------------------------
# LOGGING
# ---------------------------------------------------------------------------
import logging
logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s  %(levelname)-8s  %(message)s",
    datefmt="%H:%M:%S",
)
log = logging.getLogger("figure_s03")

# =============================================================================
# CONFIG
# =============================================================================

REPROCESS = False  # Set True to re-extract GLORYS and regenerate scatter figs;
                   # False loads saved CSV and goes straight to barchart.

MAT_DIR    = Path("/local/jbj13rpu/Documents/ROVER/from_laura/"
                  "EGC_mooring_data_LdeSteur 2/EGC_mooring_data_LdeSteur")
GLORYS_DIR = Path("/local/jbj13rpu/Documents/ROVER/glorys12_with_density")
GLORYS_GLOB = "glorys12_*_Greenland_Sea_with_density.nc"

OUT_FIG_DIR      = Path("./outputs/figures")
OUT_DATA_DIR     = Path("./outputs/processed_data/figure_s03")
OUT_METHODS_DIR  = Path("./outputs/methods")
CSV_COLLOC       = OUT_DATA_DIR / "glorys_mooring_collocated.csv"
CSV_STATS        = OUT_DATA_DIR / "mooring_validation_stats.csv"
OUT_FIG_BARCHART = OUT_FIG_DIR  / "figure_s03_FS_mooring_glorys_verification.png"

# Known NPI mooring positions (exact deployment records)
MOORING_POSITIONS = {
    "F17": (78.838, -8.107),
    "F14": (78.814, -6.503),
    "F13": (78.836, -5.000),
    "F12": (78.816, -3.990),
    "F11": (78.808, -3.053),
}

# West-to-east order for bar chart x-axis
MOORING_ORDER  = ["F17", "F14", "F13", "F12", "F11"]
MOORING_LONS   = {"F17": -8.1, "F14": -6.5, "F13": -5.0, "F12": -4.0, "F11": -3.1}

# Plotting style
COL_OBS    = "black"
COL_GLORYS = "#2E86AB"    # PSW blue
COL_REG    = "#D84315"    # regression line
COL_BIAS   = "#2E86AB"
COL_RMSE   = "#D84315"
FONTSIZE   = 11
STATS_SIZE = 9
FIG_DPI    = 600

# =============================================================================
# HELPERS: MATLAB DATE CONVERSION
# =============================================================================

def matlab_datenum_to_datetime(datenum_array):
    """
    Convert MATLAB datenum to pandas DatetimeIndex.
    Anchor: MATLAB datenum 719529 == 1970-01-01 (Unix epoch).
    Using vectorised float arithmetic avoids overflow entirely.
    """
    dn = np.asarray(datenum_array, dtype=float)
    days_since_epoch = dn - 719529.0
    timestamps = pd.to_datetime(days_since_epoch, unit="D", origin="unix")
    return timestamps


# =============================================================================
# HELPERS: LOAD .MAT MOORING FILE
# =============================================================================

def load_mat_mooring(mat_path):
    """
    Load a Laura de Steur timeseries .mat file and return monthly means.

    Expected keys: P, S, SIG, T, time
    Returns a DataFrame indexed by time (monthly means, ≥90% coverage)
    with columns: P_dbar, T_insitu, S, T_pot
    """
    log.info(f"  Loading {mat_path.name}")
    mat = sio.loadmat(str(mat_path), squeeze_me=True, struct_as_record=False)

    time_dt  = matlab_datenum_to_datetime(mat["time"])
    T_insitu = mat["T"].astype(float)
    S        = mat["S"].astype(float)
    P        = mat["P"].astype(float)   # dbar

    # Replace fill values with NaN
    for arr in (T_insitu, S, P):
        arr[np.abs(arr) > 1e10] = np.nan

    # Convert in-situ T -> potential temperature (ref pressure = 0 dbar)
    # gsw.pt_from_t(SA, t, p, p_ref); approximate SA ~ S (small error at ~35 psu)
    SA    = gsw.SA_from_SP(S, P, lon=0.0, lat=79.0)   # approximate lon/lat
    T_pot = gsw.pt_from_t(SA, T_insitu, P, p_ref=0.0)

    df = pd.DataFrame({
        "time":     time_dt,
        "P_dbar":   P,
        "T_insitu": T_insitu,
        "S":        S,
        "T_pot":    T_pot,
    })
    df = df.set_index("time")
    df = df[~df.index.isna()]
    df = df.dropna(subset=["T_pot", "S"])
    log.info(f"    {len(df)} valid raw records, "
             f"{df.index[0].strftime('%Y-%m')} – {df.index[-1].strftime('%Y-%m')}")

    # Resample to monthly means with 90% coverage threshold
    MIN_COVERAGE  = 0.90
    monthly_mean  = df.resample("MS").mean()
    monthly_count = df.resample("MS").count()
    monthly_total = df.resample("MS").size()   # all records including NaN rows

    coverage = monthly_count.div(monthly_total, axis=0)
    mask = coverage >= MIN_COVERAGE
    monthly_mean[~mask] = np.nan
    monthly_mean = monthly_mean.dropna(subset=["T_pot", "S"])

    log.info(f"    {len(monthly_mean)} monthly means retained "
             f"(>= {int(MIN_COVERAGE*100)}% coverage)")
    return monthly_mean


# =============================================================================
# HELPERS: LOAD GLORYS
# =============================================================================

_glorys_cache = {}

def load_glorys():
    if "ds" in _glorys_cache:
        return _glorys_cache["ds"]
    files = sorted(GLORYS_DIR.glob(GLORYS_GLOB))
    if not files:
        raise FileNotFoundError(f"No GLORYS files: {GLORYS_DIR / GLORYS_GLOB}")
    log.info(f"  Loading {len(files)} GLORYS files ...")
    ds = xr.open_mfdataset(
        files,
        combine="by_coords",
        data_vars="minimal",
        coords="minimal",
        compat="override",
        parallel=False,
    )
    # Standardise coord names
    rename = {}
    for old, new in [("lat", "latitude"), ("lon", "longitude")]:
        if old in ds.coords and new not in ds.coords:
            rename[old] = new
    if rename:
        ds = ds.rename(rename)
    _glorys_cache["ds"] = ds
    log.info(f"    time: {str(ds.time.values[0])[:7]} -> "
             f"{str(ds.time.values[-1])[:7]}")
    log.info(f"    depth: {len(ds.depth)} levels  "
             f"{float(ds.depth.min()):.0f}–{float(ds.depth.max()):.0f} m")
    return ds


# =============================================================================
# HELPERS: GLORYS COLLOCATION
# =============================================================================

def collocate_glorys_to_obs(df_obs, lat, lon, ds_glorys):
    """
    For each observation in df_obs, extract GLORYS T and S at (lat, lon)
    interpolated vertically to the observation's pressure (treated as depth in m).

    Matching: by year + month (GLORYS is monthly).
    Vertical: np.interp along GLORYS depth axis.
    Horizontal: nearest neighbour.

    Returns df_obs with two new columns: glorys_T, glorys_S
    """
    glat = ds_glorys.latitude.values
    glon = ds_glorys.longitude.values
    gdep = ds_glorys.depth.values   # metres

    i_lat = int(np.argmin(np.abs(glat - lat)))
    i_lon = int(np.argmin(np.abs(glon - lon)))
    log.info(f"    GLORYS nearest point: "
             f"{glat[i_lat]:.3f}°N  {glon[i_lon]:.3f}°E  "
             f"(obs: {lat:.3f}°N  {lon:.3f}°E)")

    log.info("    Extracting GLORYS profile timeseries at mooring position ...")
    T_glorys_full = ds_glorys["thetao"].isel(
        latitude=i_lat, longitude=i_lon).load().values   # (time, depth)
    S_glorys_full = ds_glorys["so"].isel(
        latitude=i_lat, longitude=i_lon).load().values   # (time, depth)

    glorys_times = pd.DatetimeIndex(ds_glorys.time.values)
    glorys_ym = {(t.year, t.month): i for i, t in enumerate(glorys_times)}

    T_collocated = np.full(len(df_obs), np.nan)
    S_collocated = np.full(len(df_obs), np.nan)

    for idx, (ts, row) in enumerate(df_obs.iterrows()):
        ym = (ts.year, ts.month)
        if ym not in glorys_ym:
            continue
        t_idx = glorys_ym[ym]

        obs_depth = float(row["P_dbar"])   # dbar ≈ m
        if np.isnan(obs_depth) or obs_depth < 0:
            continue

        t_prof = T_glorys_full[t_idx, :]
        s_prof = S_glorys_full[t_idx, :]

        valid = np.isfinite(t_prof) & np.isfinite(s_prof)
        if valid.sum() < 2:
            continue

        if obs_depth > gdep[valid].max():
            continue   # below GLORYS column — skip

        T_collocated[idx] = np.interp(obs_depth, gdep[valid], t_prof[valid])
        S_collocated[idx] = np.interp(obs_depth, gdep[valid], s_prof[valid])

    df_out = df_obs.copy()
    df_out["glorys_T"] = T_collocated
    df_out["glorys_S"] = S_collocated

    n_matched = np.sum(np.isfinite(T_collocated))
    log.info(f"    Matched {n_matched}/{len(df_obs)} obs to GLORYS timesteps")
    return df_out


# =============================================================================
# HELPERS: FIND .MAT FILE FOR MOORING
# =============================================================================

def find_mat_file(mooring_id):
    """Search MAT_DIR for a file matching timeseries_F{id}_*.mat"""
    pattern = str(MAT_DIR / f"timeseries_{mooring_id}_*.mat")
    matches = glob.glob(pattern)
    if not matches:
        log.warning(f"  No .mat file found for {mooring_id} in {MAT_DIR}")
        return None
    if len(matches) > 1:
        log.warning(f"  Multiple .mat files for {mooring_id}: {matches} — using first")
    return Path(matches[0])


# =============================================================================
# HELPERS: SCATTER FIGURES (reference — not used in paper)
# =============================================================================

def scatter_stats(obs, mod):
    """Return dict of stats for a matched obs/model pair (NaN-safe)."""
    mask = np.isfinite(obs) & np.isfinite(mod)
    if mask.sum() < 5:
        return None
    o, m = obs[mask], mod[mask]
    bias  = float(np.mean(m - o))
    rmse  = float(np.sqrt(np.mean((m - o) ** 2)))
    r, _  = stats.pearsonr(o, m)
    slope, intercept, *_ = stats.linregress(o, m)
    return dict(N=int(mask.sum()), bias=bias, rmse=rmse, r=r,
                slope=slope, intercept=intercept)


def plot_scatter_panel(ax, obs, mod, label_obs, label_mod,
                       xlabel, ylabel, color, title):
    """Draw one scatter panel with 1:1 line, regression, and stats box."""
    st = scatter_stats(np.array(obs), np.array(mod))
    if st is None:
        ax.text(0.5, 0.5, "Insufficient data", transform=ax.transAxes,
                ha="center", va="center", fontsize=FONTSIZE)
        ax.set_title(title, fontsize=FONTSIZE)
        return

    mask = np.isfinite(np.array(obs)) & np.isfinite(np.array(mod))
    o, m = np.array(obs)[mask], np.array(mod)[mask]

    ax.scatter(o, m, s=8, alpha=0.4, color=color,
               linewidths=0, rasterized=True, label="monthly obs")

    lims = [min(o.min(), m.min()), max(o.max(), m.max())]
    pad  = 0.05 * (lims[1] - lims[0])
    lims = [lims[0] - pad, lims[1] + pad]
    ax.plot(lims, lims, color="0.6", lw=1.0, ls="--", zorder=2, label="1:1")

    x_reg = np.array(lims)
    y_reg = st["slope"] * x_reg + st["intercept"]
    ax.plot(x_reg, y_reg, color=COL_REG, lw=1.4, zorder=3, label="regression")

    ax.set_xlim(lims)
    ax.set_ylim(lims)
    ax.set_xlabel(xlabel, fontsize=FONTSIZE)
    ax.set_ylabel(ylabel, fontsize=FONTSIZE)
    ax.tick_params(labelsize=FONTSIZE)
    ax.set_title(title, fontsize=FONTSIZE)
    ax.set_aspect("equal", adjustable="box")

    stats_text = (
        f"N = {st['N']}\n"
        f"Bias = {st['bias']:+.3f}\n"
        f"RMSE = {st['rmse']:.3f}\n"
        f"r = {st['r']:.3f}\n"
        f"slope = {st['slope']:.3f}"
    )
    ax.text(0.04, 0.96, stats_text, transform=ax.transAxes,
            fontsize=STATS_SIZE, va="top", ha="left",
            fontfamily="monospace",
            bbox=dict(boxstyle="round,pad=0.3", fc="white", alpha=0.8, ec="0.7"))

    ax.legend(fontsize=10, loc="lower right")


def make_scatter_figure(df_collocated, mooring_id):
    """Create and save T and S scatter figure for one mooring (reference only)."""
    fig, axes = plt.subplots(1, 2, figsize=(10, 5))
    fig.suptitle(f"GLORYS12 vs mooring {mooring_id} — T & S scatter",
                 fontsize=FONTSIZE)

    plot_scatter_panel(
        ax=axes[0],
        obs=df_collocated["T_pot"],
        mod=df_collocated["glorys_T"],
        label_obs="Obs (θ)",
        label_mod="GLORYS (thetao)",
        xlabel="Observed θ (°C)",
        ylabel="GLORYS thetao (°C)",
        color=COL_GLORYS,
        title=f"{mooring_id} — Temperature",
    )

    plot_scatter_panel(
        ax=axes[1],
        obs=df_collocated["S"],
        mod=df_collocated["glorys_S"],
        label_obs="Obs (S)",
        label_mod="GLORYS (so)",
        xlabel="Observed S (psu)",
        ylabel="GLORYS so (psu)",
        color="#F77F00",
        title=f"{mooring_id} — Salinity",
    )

    plt.tight_layout()
    out_path = OUT_FIG_DIR / f"fig_supp_mooring_scatter_{mooring_id}.png"
    fig.savefig(out_path, dpi=FIG_DPI, bbox_inches="tight")
    plt.close(fig)
    log.info(f"  Saved scatter: {out_path}")
    return out_path


# =============================================================================
# HELPERS: VALIDATION STATISTICS
# =============================================================================

def compute_stats(obs, mod):
    """N, bias, RMSE, r, p(r) for paired finite arrays."""
    o = np.asarray(obs, dtype=float)
    m = np.asarray(mod, dtype=float)
    mask = np.isfinite(o) & np.isfinite(m)
    n = int(mask.sum())
    if n < 3:
        return dict(N=n, Bias=np.nan, RMSE=np.nan, r=np.nan, p_r=np.nan)
    ov, mv = o[mask], m[mask]
    bias = float(np.mean(mv - ov))
    rmse = float(np.sqrt(np.mean((mv - ov) ** 2)))
    r, p_r = stats.pearsonr(ov, mv)
    return dict(N=n, Bias=bias, RMSE=rmse, r=float(r), p_r=float(p_r))


def compute_all_stats(df_all, moorings):
    """Compute bias/RMSE/r for T and S at each mooring. Returns DataFrame."""
    rows = []
    for mid in moorings:
        df_m = df_all[df_all["mooring"] == mid].dropna(
            subset=["T_pot", "S", "glorys_T", "glorys_S"]
        )
        if df_m.empty:
            log.warning(f"  {mid}: no matched data, skipping stats")
            continue
        for var_label, obs_col, mod_col in [
            ("T (°C)",  "T_pot", "glorys_T"),
            ("S (psu)", "S",     "glorys_S"),
        ]:
            st = compute_stats(df_m[obs_col], df_m[mod_col])
            rows.append({"Mooring": mid, "Variable": var_label, **st})
    return pd.DataFrame(rows)


def print_stats_table(df_stats):
    """Print formatted validation statistics table to stdout."""
    header = (f"{'Mooring':<8}  {'Variable':<9}  {'N':>6}  "
              f"{'Bias':>8}  {'RMSE':>8}  {'r':>7}  {'p(r)':>10}")
    sep = "-" * len(header)
    print("\n" + sep)
    print("GLORYS12 vs Fram Strait mooring observations — validation statistics")
    print("(monthly means, ≥90% data coverage; bias = GLORYS − obs)")
    print(sep)
    print(header)
    print(sep)
    prev_mooring = None
    for _, row in df_stats.iterrows():
        if prev_mooring and row["Mooring"] != prev_mooring:
            print()
        p_str = f"{row['p_r']:.2e}" if np.isfinite(row["p_r"]) else "  —"
        print(
            f"{row['Mooring']:<8}  {row['Variable']:<9}  {row['N']:>6}  "
            f"{row['Bias']:>+8.4f}  {row['RMSE']:>8.4f}  "
            f"{row['r']:>7.4f}  {p_str:>10}"
        )
        prev_mooring = row["Mooring"]
    print(sep + "\n")


# =============================================================================
# HELPERS: BAR-CHART FIGURE (Supplementary Fig S3)
# =============================================================================

def draw_bar_panel(ax, df_var, panel_label, ylabel):
    """Draw one bias/RMSE bar-chart panel (T or S)."""
    x     = np.arange(len(MOORING_ORDER))
    width = 0.35

    bias = df_var["Bias"].values.astype(float)
    rmse = df_var["RMSE"].values.astype(float)

    bars_bias = ax.bar(x - width / 2, bias, width, label="Bias",
                       color=COL_BIAS, alpha=0.85, zorder=3)
    bars_rmse = ax.bar(x + width / 2, rmse, width, label="RMSE",
                       color=COL_RMSE, alpha=0.75, zorder=3)

    ax.axhline(0, color="0.3", linewidth=0.8, linestyle="--", zorder=2)

    # Value labels on bars
    for bar in bars_bias:
        h = bar.get_height()
        va = "bottom" if h >= 0 else "top"
        offset = 0.002 if h >= 0 else -0.002
        ax.text(bar.get_x() + bar.get_width() / 2,
                h + offset, f"{h:+.3f}",
                ha="center", va=va, fontsize=8, color="0.2")

    for bar in bars_rmse:
        h = bar.get_height()
        ax.text(bar.get_x() + bar.get_width() / 2,
                h + 0.002, f"{h:.3f}",
                ha="center", va="bottom", fontsize=8, color="0.2")

    max_abs = max(np.nanmax(np.abs(bias)), np.nanmax(rmse)) * 1.45
    ax.set_ylim(-max_abs, max_abs)
    ax.set_ylabel(ylabel, fontsize=FONTSIZE)
    ax.tick_params(labelsize=FONTSIZE)
    ax.yaxis.set_major_formatter(mticker.FormatStrFormatter("%.2f"))
    ax.grid(axis="y", color="0.88", linewidth=0.5, zorder=0)
    ax.set_axisbelow(True)
    for spine in ax.spines.values():
        spine.set_visible(True)
    ax.set_title(panel_label, fontsize=FONTSIZE, loc="left", pad=4)
    ax.legend(fontsize=9, loc="lower right", frameon=True,
              framealpha=0.9, edgecolor="0.8")


def make_barchart_figure(df_stats):
    """Create and save the Supplementary Fig S3 bar-chart."""
    xticklabels = [f"{m}\n({MOORING_LONS[m]:.1f}°E)" for m in MOORING_ORDER]

    df_T = (df_stats[df_stats["Variable"].str.startswith("T")]
            .set_index("Mooring").reindex(MOORING_ORDER))
    df_S = (df_stats[df_stats["Variable"].str.startswith("S")]
            .set_index("Mooring").reindex(MOORING_ORDER))

    fig, axes = plt.subplots(2, 1, figsize=(8, 7), sharex=True)
    fig.subplots_adjust(hspace=0.12)

    draw_bar_panel(axes[0], df_T,
                   "(a) Temperature verification at ~60 m depth",
                   ylabel="Temperature (°C)")
    draw_bar_panel(axes[1], df_S,
                   "(b) Salinity verification at ~60 m depth",
                   ylabel="Salinity (psu)")

    axes[1].set_xticks(np.arange(len(MOORING_ORDER)))
    axes[1].set_xticklabels(xticklabels, fontsize=FONTSIZE)
    axes[1].set_xlabel("Mooring", fontsize=FONTSIZE)

    fig.savefig(OUT_FIG_BARCHART, dpi=FIG_DPI, bbox_inches="tight")
    plt.close(fig)
    log.info(f"  Saved: {OUT_FIG_BARCHART}")


# =============================================================================
# MAIN
# =============================================================================

def main():
    moorings = ["F11", "F12", "F13", "F14", "F17"]

    # Create output directories
    OUT_FIG_DIR.mkdir(parents=True, exist_ok=True)
    OUT_DATA_DIR.mkdir(parents=True, exist_ok=True)
    OUT_METHODS_DIR.mkdir(parents=True, exist_ok=True)

    # -------------------------------------------------------------------------
    # Step 1: Load or reprocess collocated GLORYS/mooring data
    # -------------------------------------------------------------------------
    if not REPROCESS and CSV_COLLOC.exists():
        log.info(f"Loading saved collocation CSV: {CSV_COLLOC}")
        df_all = pd.read_csv(CSV_COLLOC, parse_dates=["time"], index_col="time")
    else:
        log.info("--- Loading GLORYS12 ---")
        ds_glorys = load_glorys()

        records = []
        for mid in moorings:
            log.info(f"\n--- Processing {mid} ---")
            mat_path = find_mat_file(mid)
            if mat_path is None:
                continue
            lat, lon = MOORING_POSITIONS[mid]
            df_obs = load_mat_mooring(mat_path)
            df_col = collocate_glorys_to_obs(df_obs, lat, lon, ds_glorys)
            df_col["mooring"] = mid
            df_col["lat"]     = lat
            df_col["lon"]     = lon
            records.append(df_col)

        df_all = pd.concat(records)
        df_all.index.name = "time"
        df_all.to_csv(CSV_COLLOC)
        log.info(f"\nSaved collocation CSV: {CSV_COLLOC}")

    # -------------------------------------------------------------------------
    # Step 2: Scatter figures (reference only -- not used in paper)
    # -------------------------------------------------------------------------
    if REPROCESS:
        log.info("\n--- Scatter figures (reference) ---")
        for mid in moorings:
            df_m = df_all[df_all["mooring"] == mid].copy()
            df_m = df_m.dropna(subset=["T_pot", "S", "glorys_T", "glorys_S"])
            if df_m.empty:
                log.warning(f"  {mid}: no matched data, skipping scatter")
                continue
            log.info(f"  {mid}: {len(df_m)} matched pairs")
            make_scatter_figure(df_m, mid)

    # -------------------------------------------------------------------------
    # Step 3: Compute validation statistics
    # -------------------------------------------------------------------------
    log.info("\n--- Computing validation statistics ---")
    df_stats = compute_all_stats(df_all, moorings)
    print_stats_table(df_stats)
    df_stats.to_csv(CSV_STATS, index=False, float_format="%.6f")
    log.info(f"  Stats saved: {CSV_STATS}")

    # -------------------------------------------------------------------------
    # Step 4: Bar-chart figure (Supplementary Fig S3)
    # -------------------------------------------------------------------------
    log.info("\n--- Plotting bar-chart figure ---")
    make_barchart_figure(df_stats)

    log.info("\nDone.")


main()


11:53:52  INFO      --- Loading GLORYS12 ---
11:53:52  INFO        Loading 33 GLORYS files ...
11:53:53  INFO          time: 1993-01 -> 2025-11
11:53:53  INFO          depth: 50 levels  0–5728 m
11:53:53  INFO      
--- Processing F11 ---
11:53:53  INFO        Loading timeseries_F11_60_1997_2025.mat
11:53:53  INFO          8061 valid raw records, 1998-09 – 2024-09
11:53:53  INFO          280 monthly means retained (>= 90% coverage)
11:53:53  INFO          GLORYS nearest point: 78.833°N  -3.083°E  (obs: 78.808°N  -3.053°E)
11:53:53  INFO          Extracting GLORYS profile timeseries at mooring position ...
11:54:06  INFO          Matched 280/280 obs to GLORYS timesteps
11:54:06  INFO      
--- Processing F12 ---
11:54:06  INFO        Loading timeseries_F12_60_1997_2025.mat
11:54:06  INFO          7797 valid raw records, 1997-08 – 2025-08
11:54:06  INFO          265 monthly means retained (>= 90% coverage)
11:54:06  INFO          GLORYS nearest point: 78.833°N  -4.000°E  (obs: 78.816°N  


--------------------------------------------------------------------
GLORYS12 vs Fram Strait mooring observations — validation statistics
(monthly means, ≥90% data coverage; bias = GLORYS − obs)
--------------------------------------------------------------------
Mooring   Variable        N      Bias      RMSE        r        p(r)
--------------------------------------------------------------------
F11       T (°C)        280   -0.1464    1.1838   0.2617    9.15e-06
F11       S (psu)       280   -0.1712    0.4868   0.3159    6.64e-08

F12       T (°C)        265   +0.1332    0.5296   0.2495    4.00e-05
F12       S (psu)       265   -0.1987    0.4457   0.4838    5.90e-17

F13       T (°C)        265   +0.1732    0.2804   0.3832    1.07e-10
F13       S (psu)       265   -0.0117    0.4921   0.3585    1.87e-09

F14       T (°C)        301   +0.1355    0.2097   0.4108    1.10e-13
F14       S (psu)       301   +0.2823    0.6145   0.5010    1.56e-20

F17       T (°C)        157   +0.1294    

11:55:15  INFO        Saved: outputs/figures/figure_s03_FS_mooring_glorys_verification.png
11:55:15  INFO      
Done.
